### In this assignment, we use photos from Large-scale CelebFaces Attributes (CelebA) Dataset to train convets that can classify photos into male celebrities and female celebrities accurately.

###To make this problem more challenging, we only use 2,000 photos for training, 1,000 photos for validation, and 1,000 photos for testing.

###For the first part of the assignment, following the example of chapter 8.2. Here we train our model using only 2,000 training photos. Note we need to use data augmentation and dropout to fight overfiting. This part is worth 40 points.

###For the second part, we use a pretrained convet (VGG16) to help with our task. Do the fast feature extraction without augmentation following chapter 8.3.1. And do the fine-tuning following chapter 8.3.2. This part is worth 60 points.

###Important information before you start: all the photos are of the size (218, 178), where height = 218, and width = 178.

###Finally, we should be able to achieve 94% of accuracy following the code provided by this module. Not much changes are needed.

Before we start, we need to copy hw6.zip to Google drive. And then we need to mount the Google drive using the following statements.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


We then unzip hw6.zip.

In [2]:
!unzip /content/drive/My\ Drive/hw6.zip

Archive:  /content/drive/My Drive/hw6.zip
replace hw6/train/female/000878.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace hw6/train/female/000259.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [3]:
import keras
keras.__version__

'3.13.2'

In [4]:
import os, shutil

In [5]:
base_dir = '/content//hw6'

# Directories for our training,
# validation and test splits
train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'validation')
test_dir = os.path.join(base_dir, 'test')

# Directory with our training male pictures
train_male_dir = os.path.join(train_dir, 'male')

# Directory with our training female pictures
train_female_dir = os.path.join(train_dir, 'female')

# Directory with our validation male pictures
validation_male_dir = os.path.join(validation_dir, 'male')

# Directory with our validation female pictures
validation_female_dir = os.path.join(validation_dir, 'female')

# Directory with our validation male pictures
test_male_dir = os.path.join(test_dir, 'male')

# Directory with our validation female pictures
test_female_dir = os.path.join(test_dir, 'female')

As a sanity check, let's count how many pictures we have in each training split (train/validation/test):

In [6]:
print('total training male images:', len(os.listdir(train_male_dir)))

total training male images: 1000


In [7]:
print('total training female images:', len(os.listdir(train_female_dir)))

total training female images: 1000


In [8]:
print('total validation male images:', len(os.listdir(validation_male_dir)))

total validation male images: 500


In [9]:
print('total validation female images:', len(os.listdir(validation_female_dir)))

total validation female images: 500


In [10]:
print('total test male images:', len(os.listdir(test_male_dir)))

total test male images: 500


In [11]:
print('total test female images:', len(os.listdir(test_female_dir)))

total test female images: 500


###Finish the rest by modifying the code provided in this module.

Note not much changes expected.

Just make sure the model fine_turning.keras is trained using 50 epochs.

Below is the test accuracy obtained from one of the runs.

In total, it takes rougly 10 minutes to finish when using GPU.

In [12]:
from tensorflow.keras import layers
from tensorflow.keras.utils import image_dataset_from_directory

train_dataset = image_dataset_from_directory(
    train_dir,
    image_size=(218, 178),
    batch_size=32
)
validation_dataset = image_dataset_from_directory(
    validation_dir,
    image_size=(218, 178),
    batch_size=32
)
test_dataset = image_dataset_from_directory(
    test_dir,
    image_size=(218, 178),
    batch_size=32
)

Found 2000 files belonging to 2 classes.
Found 1000 files belonging to 2 classes.
Found 1000 files belonging to 2 classes.


In [13]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.2),
    ]
)

inputs = keras.Input(shape=(218, 178, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(1./255)(x)
x = layers.Conv2D(filters=32, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=64, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=256, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=256, kernel_size=3, activation="relu")(x)
x = layers.Flatten()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    loss="binary_crossentropy",
    optimizer="rmsprop",
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="convnet_from_scratch_with_augmentation.keras",
        save_best_only=True,
        monitor="val_loss"
    )
]

history_scratch = model.fit(
    train_dataset,
    epochs=50,
    validation_data=validation_dataset,
    callbacks=callbacks
)

# Evaluate on test set
test_model = keras.models.load_model("convnet_from_scratch_with_augmentation.keras")
test_loss, test_acc = test_model.evaluate(test_dataset)
print(f"Test accuracy: {test_acc:.3f}")

Epoch 1/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 13s 74ms/step - accuracy: 0.5755 - loss: 0.6763 - val_accuracy: 0.5660 - val_loss: 0.6785
Epoch 2/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 54ms/step - accuracy: 0.7055 - loss: 0.5974 - val_accuracy: 0.7210 - val_loss: 0.5568
Epoch 3/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 54ms/step - accuracy: 0.7385 - loss: 0.5443 - val_accuracy: 0.6470 - val_loss: 0.7166
Epoch 4/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step - accuracy: 0.7575 - loss: 0.5073 - val_accuracy: 0.8200 - val_loss: 0.3845
Epoch 5/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 48ms/step - accuracy: 0.8125 - loss: 0.4204 - val_accuracy: 0.7980 - val_loss: 0.4936
Epoch 6/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step - accuracy: 0.8350 - loss: 0.3576 - val_accuracy: 0.8190 - val_loss: 0.3692
Epoch 7/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 65ms/step - accuracy: 0.8710 - loss: 0.3068 - val_accuracy: 0.8960 - val_loss: 0.2580
Epoch 8/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 49ms/step - accuracy: 0.8885 - loss: 0.2802 - val_accuracy: 0.8960 - 

In [14]:
import numpy as np

conv_base = keras.applications.vgg16.VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(218, 178, 3)
)

def get_features_and_labels(dataset):
    all_features = []
    all_labels = []
    for images, labels in dataset:
        preprocessed_images = keras.applications.vgg16.preprocess_input(images)
        features = conv_base.predict(preprocessed_images)
        all_features.append(features)
        all_labels.append(labels)
    return np.concatenate(all_features), np.concatenate(all_labels)

train_features, train_labels = get_features_and_labels(train_dataset)
val_features, val_labels = get_features_and_labels(validation_dataset)
test_features, test_labels = get_features_and_labels(test_dataset)

# Check output shape of conv_base
inputs = keras.Input(shape=train_features.shape[1:])
x = layers.Flatten()(inputs)
x = layers.Dense(256)(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model_fe = keras.Model(inputs, outputs)
model_fe.compile(
    loss="binary_crossentropy",
    optimizer="rmsprop",
    metrics=["accuracy"]
)

callbacks_fe = [
    keras.callbacks.ModelCheckpoint(
        filepath="feature_extraction.keras",
        save_best_only=True,
        monitor="val_loss"
    )
]

history_fe = model_fe.fit(
    train_features, train_labels,
    epochs=20,
    validation_data=(val_features, val_labels),
    callbacks=callbacks_fe
)

test_model_fe = keras.models.load_model("feature_extraction.keras")
test_loss, test_acc = test_model_fe.evaluate(test_features, test_labels)
print(f"Test accuracy: {test_acc:.3f}")

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 17s 17s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step
1/1 ━━━━━━━━━━

In [15]:
conv_base.trainable = False

inputs = keras.Input(shape=(218, 178, 3))
x = data_augmentation(inputs)
x = keras.applications.vgg16.preprocess_input(x)
x = conv_base(x)
x = layers.Flatten()(x)
x = layers.Dense(256)(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model_ft = keras.Model(inputs, outputs)

# Pre-train the classifier head
model_ft.compile(
    loss="binary_crossentropy",
    optimizer="rmsprop",
    metrics=["accuracy"]
)
model_ft.fit(
    train_dataset,
    epochs=10,
    validation_data=validation_dataset
)

# Unfreeze top 4 layers and fine-tune for 50 epochs
conv_base.trainable = True
for layer in conv_base.layers[:-4]:
    layer.trainable = False

model_ft.compile(
    loss="binary_crossentropy",
    optimizer=keras.optimizers.RMSprop(learning_rate=1e-5),
    metrics=["accuracy"]
)

callbacks_ft = [
    keras.callbacks.ModelCheckpoint(
        filepath="fine_tuning.keras",
        save_best_only=True,
        monitor="val_loss"
    )
]

history_ft = model_ft.fit(
    train_dataset,
    epochs=50,
    validation_data=validation_dataset,
    callbacks=callbacks_ft
)

Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 22s 234ms/step - accuracy: 0.8290 - loss: 11.5285 - val_accuracy: 0.8970 - val_loss: 7.1434
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 165ms/step - accuracy: 0.8885 - loss: 6.1692 - val_accuracy: 0.8240 - val_loss: 14.1939
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 166ms/step - accuracy: 0.8965 - loss: 5.2749 - val_accuracy: 0.9020 - val_loss: 5.8150
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 165ms/step - accuracy: 0.9065 - loss: 4.1382 - val_accuracy: 0.8680 - val_loss: 7.6874
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 161ms/step - accuracy: 0.9130 - loss: 3.2785 - val_accuracy: 0.8870 - val_loss: 6.0648
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 160ms/step - accuracy: 0.9130 - loss: 3.4097 - val_accuracy: 0.8950 - val_loss: 5.5503
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 162ms/step - accuracy: 0.9230 - loss: 2.3066 - val_accuracy: 0.8460 - val_loss: 7.2274
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 162ms/step - accuracy: 0.9255 - loss: 1.7435 - val_ac

In [16]:
model = keras.models.load_model("fine_tuning.keras")
test_loss, test_acc = model.evaluate(test_dataset)
print(f"Test accuracy: {test_acc:.3f}")

32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 102ms/step - accuracy: 0.9400 - loss: 0.9636
Test accuracy: 0.940
